# Módulo 09 · Evaluación

Se mide la calidad de la recuperación con 30 preguntas de referencia y se comparan
las configuraciones disponibles.

In [1]:
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
DATA_DIR = ROOT / "data" / "novatech"

In [2]:
from collections import Counter

from fundamenta.evaluation import load_questions

questions = load_questions(ROOT / "evaluacion" / "preguntas.jsonl")
print(Counter(q.category for q in questions))
for q in questions[:2] + questions[15:16] + questions[-1:]:
    print(f"{q.id} [{q.category}] {q.question}  ->  {q.source}: {q.evidence}")

Counter({'directa': 15, 'parafraseada': 12, 'sin_respuesta': 3})
D01 [directa] ¿Cuántos días de vacaciones al año tiene cada persona?  ->  manual-del-empleado.md: ['23 días laborables']
D02 [directa] ¿Con cuánta antelación hay que solicitar las vacaciones?  ->  manual-del-empleado.md: ['de antelación']
P01 [parafraseada] Me han robado el laptop, ¿qué tengo que hacer?  ->  faq-soporte-it.txt: ['bloqueará el equipo']
N03 [sin_respuesta] ¿A cuánto cotizan las acciones de la empresa?  ->  None: []


## Comparación de configuraciones

- **Tasa de acierto (recall@4):** proporción de preguntas en las que el fragmento con
  la respuesta está entre los cuatro primeros.
- **MRR:** media de 1/posición del primer fragmento relevante (1 = siempre el primero).

In [3]:
from fundamenta.chunking import chunk_documents
from fundamenta.embeddings import LSAEmbedder, SentenceTransformerEmbedder
from fundamenta.evaluation import evaluate_retrieval
from fundamenta.loaders import load_directory
from fundamenta.retriever import Retriever

chunks = chunk_documents(load_directory(DATA_DIR))
retrievers = {
    "LSA": Retriever(chunks, embedder=LSAEmbedder()),
    "E5": Retriever(chunks, embedder=SentenceTransformerEmbedder()),
}

reports = {}
print(f"{'Configuración':28} {'Acierto@4':>9} {'MRR':>6} {'Directas':>9} {'Parafraseadas':>13}")
for name, mode, embedder in [
    ("Léxica (BM25)", "lexical", "LSA"),
    ("Semántica (LSA)", "semantic", "LSA"),
    ("Híbrida (BM25 + LSA)", "hybrid", "LSA"),
    ("Semántica (E5)", "semantic", "E5"),
    ("Híbrida (BM25 + E5)", "hybrid", "E5"),
]:
    retriever = retrievers[embedder]
    retriever.mode = mode
    report = evaluate_retrieval(retriever.search, questions, k=4)
    reports[name] = report
    cat = report.by_category
    print(
        f"{name:28} {report.hit_rate:>9.0%} {report.mrr:>6.3f} "
        f"{cat['directa']:>9.0%} {cat['parafraseada']:>13.0%}"
    )

Configuración                Acierto@4    MRR  Directas Parafraseadas
Léxica (BM25)                      78%  0.759      100%           50%
Semántica (LSA)                    78%  0.722      100%           50%
Híbrida (BM25 + LSA)               78%  0.722      100%           50%


Semántica (E5)                     96%  0.824      100%           92%


Híbrida (BM25 + E5)                93%  0.824      100%           83%


## Análisis de errores

In [4]:
by_id = {q.id: q for q in questions}
for name in ("Léxica (BM25)", "Híbrida (BM25 + E5)"):
    print(f"{name}: fallos {reports[name].misses}")

question = by_id["P03"]
retriever = retrievers["E5"]
retriever.mode = "hybrid"
print(f"\n{question.id}: {question.question}")
print(f"Esperado: {question.source} con {question.evidence}")
for result in retriever.search(question.question, k=4):
    doc = result.document
    print(f"  {result.rank}. {doc.source} | {doc.text[:70].replace(chr(10), ' ')}…")

Léxica (BM25): fallos ['P01', 'P03', 'P04', 'P06', 'P09', 'P10']
Híbrida (BM25 + E5): fallos ['P03', 'P04']

P03: ¿Ayuda la empresa con los gastos de deporte?
Esperado: beneficios-2026.pdf con ['30 € al mes']
  1. politica-de-gastos-y-viajes.md | ## Principios generales  NovaTech cubre los gastos razonables en los q…
  2. politica-de-gastos-y-viajes.md | # Política de gastos y viajes  > Documento interno de NovaTech (empres…
  3. beneficios-2026.pdf | Beneficios para empleados 2026 NovaTech (empresa ficticia) - Documento…
  4. politica-de-gastos-y-viajes.md | ## Aprobación de viajes  - Los viajes nacionales los aprueba tu respon…


## Evaluación de respuestas

La evaluación de respuestas requiere un LLM y supone una llamada por pregunta:

```bash
fundamenta eval --embedder semantic --llm claude --answers
```

Se aplican dos criterios automáticos: que la respuesta contenga los datos esperados y
que el sistema se abstenga en las preguntas sin respuesta. Opcionalmente,
`judge_faithfulness` utiliza un LLM como evaluador para detectar afirmaciones no
respaldadas por las fuentes.

In [5]:
from fundamenta.evaluation import answer_is_correct
from fundamenta.generation import NO_ANSWER, Answer

d01, n01 = by_id["D01"], by_id["N01"]
print(answer_is_correct(Answer(d01.question, "Dispone de 23 días laborables [1]."), d01))
print(answer_is_correct(Answer(d01.question, "Dispone de 22 días laborables [1]."), d01))
print(answer_is_correct(Answer(n01.question, NO_ANSWER), n01))

True
False
True
